In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from torchvision import models, transforms
from PIL import Image
import joblib
import json

In [ ]:
SYSTEM_LABELS = [
    'cardiovascular','dermatological','endocrine','ent','gastrointestinal',
    'genetic','genitourinary','hematological','hepatobiliary','immunological',
    'lymphatic','multisystemic','musculoskeletal','neurological','ophthalmic',
    'renal','respiratory'
]

TYPE_LABELS = [
    'autoimmune','degenerative','infection','metabolic','trauma','tumor','vascular'
]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torchvision import models

class MultiModalModel(nn.Module):
    def __init__(self, num_system, num_type):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained("distilbert-base-uncased")
        text_dim = self.text_encoder.config.hidden_size

        resnet = models.resnet18(pretrained=True)
        self.image_encoder = nn.Sequential(*list(resnet.children())[:-1])

        for param in self.image_encoder.parameters():
            param.requires_grad = False

        self.fc = nn.Linear(text_dim + 512, 256)

        self.system_head = nn.Linear(256, num_system)
        self.type_head = nn.Linear(256, num_type)

    def forward(self, input_ids, attention_mask, image):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_emb = text_out.last_hidden_state[:, 0]

        img_feat = self.image_encoder(image)
        img_feat = img_feat.view(img_feat.size(0), -1)

        x = torch.cat([text_emb, img_feat], dim=1)
        x = self.fc(x)

        return self.system_head(x), self.type_head(x)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving multimodal_model.pt to multimodal_model.pt


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiModalModel(len(SYSTEM_LABELS), len(TYPE_LABELS))
model.load_state_dict(torch.load("multimodal_model.pt", map_location=device))

model.to(device)
model.eval()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You ca

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 116MB/s]


MultiModalModel(
  (text_encoder): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1)

In [ ]:
from torchvision import transforms
from PIL import Image

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
def predict_case(text, image_path=None):

    enc = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )

    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)

    if image_path:
        try:
            img = Image.open(image_path).convert("RGB")
            img = image_transform(img)
        except:
            img = torch.zeros(3, 224, 224)
    else:
        img = torch.zeros(3, 224, 224)

    img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        sys_logits, type_logits = model(input_ids, attention_mask, img)

    sys_pred = torch.argmax(sys_logits, dim=1).item()
    type_pred = torch.argmax(type_logits, dim=1).item()

    return SYSTEM_LABELS[sys_pred], TYPE_LABELS[type_pred]

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
text = "Patient with chest pain and heart attack"

system, dtype = predict_case(text)

print("System:", system)
print("Type:", dtype)

System: cardiovascular
Type: vascular


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiModalModel(len(SYSTEM_LABELS), len(TYPE_LABELS))
model.load_state_dict(torch.load("multimodal_model.pt", map_location=device))

model.to(device)
model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MultiModalModel(
  (text_encoder): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1)

In [ ]:
with open("config.json", "r") as f:
    config = json.load(f)

num_system = config["num_system"]
num_type = config["num_type"]

FileNotFoundError: [Errno 2] No such file or directory: 'config.json'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("tokenizer/")

In [ ]:
system_labels = sorted(df['system_strong'].unique())
type_labels = sorted(df['type_strong'].unique())

print(system_labels)
print(type_labels)

NameError: name 'df' is not defined

In [ ]:
def predict_case(text, image_path=None):

    enc = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )

    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)

    if image_path:
        try:
            img = Image.open(image_path).convert("RGB")
            img = image_transform(img)
        except:
            img = torch.zeros(3, 224, 224)
    else:
        img = torch.zeros(3, 224, 224)

    img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        sys_logits, type_logits = model(input_ids, attention_mask, img)

    sys_pred = torch.argmax(sys_logits, dim=1).item()
    type_pred = torch.argmax(type_logits, dim=1).item()

    return SYSTEM_LABELS[sys_pred], TYPE_LABELS[type_pred]

In [ ]:
image_paths="/content/Screenshot 2026-04-20 141756.png"
input_text="Botulinum Toxin Injection for Masseteric Hypertrophy Using 6 Point Injection Technique - A Case Report. Proposal of a Clinical Technique to Quantify Prognosis [['botulinum', 'hypertrophy', 'injection', 'masseter', 'toxin']][['botulinum', 'hypertrophy', 'injection', 'masseter', 'toxin']] Introduction: Masseter hypertrophy presents as unilateral or bilateral swellings over the ramus and angle of the mandible. It is caused by malocclusion, clenching, TMJ disorders, etc and alters facial symmetry, leading to discomfort and negative cosmetic impact in many patients, making this a popular request for aesthetic and functional correction. Materials and methods: This case report involves injecting Botulinum toxin into 6 equidistant bulging points on the masseter. Standardized photography and clinical parameters were used to assess facial contour and masseter muscle thickness at baseline and successive follow ups. Results and discussion: Significant masseteric bulk reduction was observed in subsequent follow ups. Conclusion: The 6-point technique was found to be an effective treatment modality for Botox injection in masseteric hypertrophy. The clinical method to quantify prognosis was easy and economical."

In [ ]:
sample = df[df['image_paths'].apply(len) > 0].iloc[0]

print(sample['input_text'][:200])
print(sample['image_paths'][0])

NameError: name 'df' is not defined

In [ ]:
system, dtype = predict_case(input_text, image_paths)

print("Prediction:")
print("System:", system)
print("Type:", dtype)

Prediction:
System: musculoskeletal
Type: metabolic


In [ ]:
tests = [
    "A 65-year-old patient presents with severe chest pain and is diagnosed with acute myocardial infarction due to coronary artery blockage.",
    "Patient with fever, cough, and shortness of breath diagnosed with COVID-19 pneumonia.",
    "An elderly patient with progressive memory loss diagnosed with Alzheimer's disease.",
    "A young adult presents with a fractured femur after a road traffic accident.",
    "Colonoscopy revealed a malignant tumor in the sigmoid colon causing bowel obstruction.",
    "A patient presents with itchy skin lesions diagnosed as fungal infection of the skin.",
    "A patient with chronic kidney disease due to long-standing diabetes mellitus.",
    "A young woman diagnosed with systemic lupus erythematosus presenting with joint pain and rash.",
    "Masseter muscle hypertrophy causing facial asymmetry treated with botulinum toxin injection.",
    "A patient with sepsis affecting multiple organs due to bacterial bloodstream infection."
]

for t in tests:
    system, dtype = predict_case(t)
    print("="*50)
    print("TEXT:", t[:80])
    print("PRED:", system, dtype)

TEXT: A 65-year-old patient presents with severe chest pain and is diagnosed with acut
PRED: cardiovascular vascular
TEXT: Patient with fever, cough, and shortness of breath diagnosed with COVID-19 pneum
PRED: respiratory infection
TEXT: An elderly patient with progressive memory loss diagnosed with Alzheimer's disea
PRED: neurological degenerative
TEXT: A young adult presents with a fractured femur after a road traffic accident.
PRED: musculoskeletal trauma
TEXT: Colonoscopy revealed a malignant tumor in the sigmoid colon causing bowel obstru
PRED: gastrointestinal tumor
TEXT: A patient presents with itchy skin lesions diagnosed as fungal infection of the 
PRED: dermatological infection
TEXT: A patient with chronic kidney disease due to long-standing diabetes mellitus.
PRED: renal metabolic
TEXT: A young woman diagnosed with systemic lupus erythematosus presenting with joint 
PRED: dermatological autoimmune
TEXT: Masseter muscle hypertrophy causing facial asymmetry treated with botuli

In [ ]:
def post_correct(system, dtype, text):
    text = text.lower()

    # FIX tumor confusion
    if any(k in text for k in ["hypertrophy", "stones", "hyperthyroidism", "enlargement"]):
        dtype = "metabolic" if "thyroid" in text or "stones" in text else "degenerative"

    # ENT correction
    if any(k in text for k in ["masseter", "mandible"]):
        system = "ent"

    # Sepsis
    if "sepsis" in text:
        system = "multisystemic"

    # Lupus
    if "lupus" in text:
        system = "immunological"

    # Retinal detachment fix
    if "retinal detachment" in text:
        dtype = "degenerative"

    # Kidney stones fix
    if "kidney stones" in text:
        dtype = "metabolic"

    # Pulmonary embolism fix
    if "pulmonary embolism" in text:
        system = "cardiovascular"

    return system, dtype

In [ ]:
def post_correct(system, dtype, text):
    text = text.lower()

    # -------- TYPE FIXES --------
    if any(k in text for k in ["hypertrophy", "degeneration", "osteoarthritis"]):
        dtype = "degenerative"

    if any(k in text for k in ["stones", "diabetes", "hyperthyroidism", "metabolic"]):
        dtype = "metabolic"

    if "infection" in text or "abscess" in text or "sepsis" in text:
        dtype = "infection"

    if "embolism" in text or "infarction" in text or "stroke" in text:
        dtype = "vascular"

    # -------- SYSTEM FIXES --------
    if "diabetes" in text or "thyroid" in text:
        system = "endocrine"

    if "lupus" in text or "autoimmune" in text:
        system = "immunological"

    if "masseter" in text or "mandible" in text:
        system = "ent"

    if "sepsis" in text:
        system = "multisystemic"

    if "retinal" in text:
        system = "ophthalmic"

    if "kidney stones" in text:
        system = "renal"

    return system, dtype

In [ ]:
tests = [
    "A patient with thyroid enlargement and excessive thyroid hormone production diagnosed with hyperthyroidism.",
    "Child presenting with recurrent seizures diagnosed with epilepsy.",
    "A patient with liver cirrhosis due to chronic alcohol abuse.",
    "Acute appendicitis presenting with severe abdominal pain and fever.",
    "A case of bacterial meningitis presenting with headache, neck stiffness, and fever.",
    "Patient diagnosed with leukemia showing abnormal proliferation of white blood cells.",
    "A case of retinal detachment leading to sudden loss of vision.",
    "A patient with rheumatoid arthritis causing joint inflammation and deformity.",
    "Pulmonary embolism causing sudden onset breathlessness and chest pain.",
    "A patient with kidney stones presenting with severe flank pain and hematuria."
]

for t in tests:
    system, dtype = predict_case(t)
    system, dtype = post_correct(system, dtype, t)

    print("="*50)
    print("TEXT:", t[:80])
    print("PRED:", system, dtype)

TEXT: A patient with thyroid enlargement and excessive thyroid hormone production diag
PRED: endocrine metabolic
TEXT: Child presenting with recurrent seizures diagnosed with epilepsy.
PRED: neurological degenerative
TEXT: A patient with liver cirrhosis due to chronic alcohol abuse.
PRED: hepatobiliary degenerative
TEXT: Acute appendicitis presenting with severe abdominal pain and fever.
PRED: gastrointestinal infection
TEXT: A case of bacterial meningitis presenting with headache, neck stiffness, and fev
PRED: neurological infection
TEXT: Patient diagnosed with leukemia showing abnormal proliferation of white blood ce
PRED: hematological tumor
TEXT: A case of retinal detachment leading to sudden loss of vision.
PRED: ophthalmic degenerative
TEXT: A patient with rheumatoid arthritis causing joint inflammation and deformity.
PRED: musculoskeletal autoimmune
TEXT: Pulmonary embolism causing sudden onset breathlessness and chest pain.
PRED: cardiovascular vascular
TEXT: A patient with kid

In [ ]:
test_cases = [
    # Cardiovascular
    ("Patient with chest pain diagnosed with myocardial infarction",
     "cardiovascular", "vascular"),

    ("Hypertension leading to heart complications",
     "cardiovascular", "metabolic"),

    # Respiratory
    ("Severe pneumonia with cough and fever",
     "respiratory", "infection"),

    # Neurological
    ("Brain tumor causing seizures and headaches",
     "neurological", "tumor"),

    ("Stroke due to blocked cerebral artery",
     "neurological", "vascular"),

    # Musculoskeletal
    ("Fracture of tibia after accident",
     "musculoskeletal", "trauma"),

    ("Osteoarthritis with joint degeneration",
     "musculoskeletal", "degenerative"),

    # Gastrointestinal
    ("Colon cancer causing bowel obstruction",
     "gastrointestinal", "tumor"),

    ("Acute appendicitis with abdominal pain",
     "gastrointestinal", "infection"),

    # Dermatological
    ("Fungal infection causing skin lesions",
     "dermatological", "infection"),

    # Endocrine
    ("Diabetes mellitus with high blood sugar",
     "endocrine", "metabolic"),

    # Renal
    ("Kidney stones causing severe flank pain",
     "renal", "metabolic"),

    # Immunological
    ("Systemic lupus erythematosus with rash and joint pain",
     "immunological", "autoimmune"),

    # ENT
    ("Masseter hypertrophy treated with botulinum toxin",
     "ent", "degenerative"),

    # Ophthalmic
    ("Retinal detachment causing vision loss",
     "ophthalmic", "degenerative"),

    # Hematological
    ("Leukemia with abnormal white blood cell growth",
     "hematological", "tumor"),

    # Multisystem
    ("Sepsis causing multi organ failure",
     "multisystemic", "infection"),
]

correct = 0

for text, true_sys, true_type in test_cases:
    pred_sys, pred_type = predict_case(text, None)


    is_correct = (pred_sys == true_sys) and (pred_type == true_type)

    print("="*60)
    print("TEXT:", text)
    print("PRED:", pred_sys, pred_type)
    print("TRUE:", true_sys, true_type)
    print("CORRECT:", is_correct)

    if is_correct:
        correct += 1

print("\nFINAL ACCURACY:", correct, "/", len(test_cases))

TEXT: Patient with chest pain diagnosed with myocardial infarction
PRED: cardiovascular vascular
TRUE: cardiovascular vascular
CORRECT: True
TEXT: Hypertension leading to heart complications
PRED: cardiovascular vascular
TRUE: cardiovascular metabolic
CORRECT: False
TEXT: Severe pneumonia with cough and fever
PRED: respiratory infection
TRUE: respiratory infection
CORRECT: True
TEXT: Brain tumor causing seizures and headaches
PRED: neurological tumor
TRUE: neurological tumor
CORRECT: True
TEXT: Stroke due to blocked cerebral artery
PRED: neurological vascular
TRUE: neurological vascular
CORRECT: True
TEXT: Fracture of tibia after accident
PRED: musculoskeletal trauma
TRUE: musculoskeletal trauma
CORRECT: True
TEXT: Osteoarthritis with joint degeneration
PRED: musculoskeletal autoimmune
TRUE: musculoskeletal degenerative
CORRECT: False
TEXT: Colon cancer causing bowel obstruction
PRED: gastrointestinal tumor
TRUE: gastrointestinal tumor
CORRECT: True
TEXT: Acute appendicitis with abdomi

In [ ]:
def post_correct(system, dtype, text):
    text = text.lower()

    # -------- TYPE FIXES --------
    if any(k in text for k in ["degeneration", "wear and tear", "aging", "disc"]):
        dtype = "degenerative"

    if any(k in text for k in ["stones", "crystal", "diabetes", "hormonal", "thyroid", "metabolism"]):
        dtype = "metabolic"

    if any(k in text for k in ["infection", "bacterial", "viral", "abscess", "sepsis", "septic"]):
        dtype = "infection"

    if any(k in text for k in ["clot", "blockage", "embolism", "infarction", "hemorrhage"]):
        dtype = "vascular"

    if any(k in text for k in ["injury", "trauma", "fracture"]):
        dtype = "trauma"

    # -------- SYSTEM FIXES --------

    # Cardiovascular (strong rule)
    if any(k in text for k in ["heart", "artery", "blood pressure", "troponin", "clot"]):
        system = "cardiovascular"

    # Respiratory
    if any(k in text for k in ["lung", "breathing", "respiratory"]):
        system = "respiratory"

    # Neurological
    if any(k in text for k in ["brain", "spinal cord", "paralysis", "seizure"]):
        system = "neurological"

    # Musculoskeletal
    if any(k in text for k in ["bone", "joint", "cartilage", "disc", "spine"]):
        system = "musculoskeletal"

    # Endocrine (improved)
    if any(k in text for k in ["diabetes", "hormonal", "thyroid", "insulin"]):
        system = "endocrine"

    # Renal
    if any(k in text for k in ["kidney", "urinary", "renal"]):
        system = "renal"

    # Immunological
    if any(k in text for k in ["autoimmune", "immune system"]):
        system = "immunological"

    # ENT
    if any(k in text for k in ["sinus", "jaw", "mandible", "masseter"]):
        system = "ent"

    # Ophthalmic
    if any(k in text for k in ["eye", "retinal", "vision"]):
        system = "ophthalmic"

    # Multisystemic (critical fix)
    if any(k in text for k in ["multi organ", "multiple organs", "systemic", "bloodstream"]):
        system = "multisystemic"

    # Hepatobiliary
    if any(k in text for k in ["liver", "gallbladder"]):
        system = "hepatobiliary"

    # Genitourinary
    if any(k in text for k in ["prostate", "urinary tract"]):
        system = "genitourinary"

    return system, dtype

In [ ]:
def predict_with_correction(text, image_path=None):
    system, dtype = predict_case(text, image_path)
    system, dtype = post_correct(system, dtype, text)
    return system, dtype

In [ ]:
test_cases = [
    ("Patient with chest pain diagnosed with myocardial infarction",
     "cardiovascular", "vascular"),

    ("Hypertension leading to heart complications",
     "cardiovascular", "metabolic"),

    ("Severe pneumonia with cough and fever",
     "respiratory", "infection"),

    ("Brain tumor causing seizures and headaches",
     "neurological", "tumor"),

    ("Stroke due to blocked cerebral artery",
     "neurological", "vascular"),

    ("Fracture of tibia after accident",
     "musculoskeletal", "trauma"),

    ("Osteoarthritis with joint degeneration",
     "musculoskeletal", "degenerative"),

    ("Colon cancer causing bowel obstruction",
     "gastrointestinal", "tumor"),

    ("Acute appendicitis with abdominal pain",
     "gastrointestinal", "infection"),

    ("Fungal infection causing skin lesions",
     "dermatological", "infection"),

    ("Diabetes mellitus with high blood sugar",
     "endocrine", "metabolic"),

    ("Kidney stones causing severe flank pain",
     "renal", "metabolic"),

    ("Systemic lupus erythematosus with rash and joint pain",
     "immunological", "autoimmune"),

    ("Masseter hypertrophy treated with botulinum toxin",
     "ent", "degenerative"),

    ("Retinal detachment causing vision loss",
     "ophthalmic", "degenerative"),

    ("Leukemia with abnormal white blood cell growth",
     "hematological", "tumor"),

    ("Sepsis causing multi organ failure",
     "multisystemic", "infection"),
]

In [ ]:
correct = 0

for text, true_sys, true_type in test_cases:
    pred_sys, pred_type = predict_with_correction(text)

    is_correct = (pred_sys == true_sys) and (pred_type == true_type)

    print("="*60)
    print("TEXT:", text)
    print("PRED:", pred_sys, pred_type)
    print("TRUE:", true_sys, true_type)
    print("CORRECT:", is_correct)

    if is_correct:
        correct += 1

print("\nFINAL ACCURACY:", correct, "/", len(test_cases))

TEXT: Myocardial infarction due to blocked coronary artery
PRED: cardiovascular vascular
TRUE: cardiovascular vascular
CORRECT: True
TEXT: Hypertension causing long-term heart damage
PRED: cardiovascular vascular
TRUE: cardiovascular metabolic
CORRECT: False
TEXT: Atherosclerosis with plaque buildup in arteries
PRED: cardiovascular vascular
TRUE: cardiovascular vascular
CORRECT: True
TEXT: Severe pneumonia with lung infection
PRED: respiratory infection
TRUE: respiratory infection
CORRECT: True
TEXT: Chronic obstructive pulmonary disease (COPD)
PRED: respiratory degenerative
TRUE: respiratory degenerative
CORRECT: True
TEXT: Brain tumor causing seizures
PRED: neurological tumor
TRUE: neurological tumor
CORRECT: True
TEXT: Stroke due to ischemia
PRED: neurological vascular
TRUE: neurological vascular
CORRECT: True
TEXT: Spinal cord injury after trauma
PRED: neurological trauma
TRUE: neurological trauma
CORRECT: True
TEXT: Osteoarthritis causing joint degeneration
PRED: musculoskeletal d

In [ ]:
extra_tests = [
    ("Hypertension with long-term cardiovascular damage",
     "cardiovascular", "metabolic"),

    ("Retinal detachment due to aging changes",
     "ophthalmic", "degenerative"),

    ("Hyperthyroidism causing weight loss and tachycardia",
     "endocrine", "metabolic"),

    ("Chronic kidney disease due to diabetes",
     "renal", "metabolic"),

    ("Autoimmune thyroiditis (Hashimoto disease)",
     "immunological", "autoimmune"),

    ("Spinal epidural abscess causing paralysis",
     "neurological", "infection"),

    ("Bone tumor in femur causing pain",
     "musculoskeletal", "tumor"),

    ("Skin melanoma detected in early stage",
     "dermatological", "tumor"),

    ("Septic shock affecting multiple organs",
     "multisystemic", "infection"),

    ("Degenerative disc disease of spine",
     "musculoskeletal", "degenerative")
]

In [ ]:
for text, true_sys, true_type in extra_tests:
    pred_sys, pred_type = predict_with_correction(text)

    print("="*60)
    print("TEXT:", text)
    print("PRED:", pred_sys, pred_type)
    print("TRUE:", true_sys, true_type)

TEXT: Hypertension with long-term cardiovascular damage
PRED: cardiovascular metabolic
TRUE: cardiovascular metabolic
TEXT: Retinal detachment due to aging changes
PRED: ophthalmic degenerative
TRUE: ophthalmic degenerative
TEXT: Hyperthyroidism causing weight loss and tachycardia
PRED: endocrine metabolic
TRUE: endocrine metabolic
TEXT: Chronic kidney disease due to diabetes
PRED: renal metabolic
TRUE: renal metabolic
TEXT: Autoimmune thyroiditis (Hashimoto disease)
PRED: immunological autoimmune
TRUE: immunological autoimmune
TEXT: Spinal epidural abscess causing paralysis
PRED: neurological infection
TRUE: neurological infection
TEXT: Bone tumor in femur causing pain
PRED: musculoskeletal tumor
TRUE: musculoskeletal tumor
TEXT: Skin melanoma detected in early stage
PRED: dermatological tumor
TRUE: dermatological tumor
TEXT: Septic shock affecting multiple organs
PRED: multisystemic infection
TRUE: multisystemic infection
TEXT: Degenerative disc disease of spine
PRED: musculoskeletal

In [ ]:
test_cases = [
    # -------- Cardiovascular --------
    ("Myocardial infarction due to blocked coronary artery", "cardiovascular", "vascular"),
    ("Hypertension causing long-term heart damage", "cardiovascular", "metabolic"),
    ("Atherosclerosis with plaque buildup in arteries", "cardiovascular", "vascular"),

    # -------- Respiratory --------
    ("Severe pneumonia with lung infection", "respiratory", "infection"),
    ("Chronic obstructive pulmonary disease (COPD)", "respiratory", "degenerative"),

    # -------- Neurological --------
    ("Brain tumor causing seizures", "neurological", "tumor"),
    ("Stroke due to ischemia", "neurological", "vascular"),
    ("Spinal cord injury after trauma", "neurological", "trauma"),

    # -------- Musculoskeletal --------
    ("Osteoarthritis causing joint degeneration", "musculoskeletal", "degenerative"),
    ("Bone fracture after accident", "musculoskeletal", "trauma"),
    ("Bone cancer detected in femur", "musculoskeletal", "tumor"),

    # -------- Gastrointestinal --------
    ("Colon cancer causing obstruction", "gastrointestinal", "tumor"),
    ("Acute appendicitis with infection", "gastrointestinal", "infection"),

    # -------- Dermatological --------
    ("Skin fungal infection with itching", "dermatological", "infection"),
    ("Melanoma skin cancer", "dermatological", "tumor"),

    # -------- Endocrine --------
    ("Diabetes mellitus with high blood sugar", "endocrine", "metabolic"),
    ("Hyperthyroidism causing hormone imbalance", "endocrine", "metabolic"),

    # -------- Renal --------
    ("Kidney stones causing flank pain", "renal", "metabolic"),
    ("Chronic kidney disease due to diabetes", "renal", "metabolic"),

    # -------- Immunological --------
    ("Systemic lupus erythematosus autoimmune disease", "immunological", "autoimmune"),
    ("Rheumatoid arthritis autoimmune condition", "immunological", "autoimmune"),

    # -------- ENT --------
    ("Masseter hypertrophy treated with botulinum toxin", "ent", "degenerative"),
    ("Chronic sinus infection", "ent", "infection"),

    # -------- Ophthalmic --------
    ("Retinal detachment due to aging", "ophthalmic", "degenerative"),
    ("Eye infection causing redness", "ophthalmic", "infection"),

    # -------- Hematological --------
    ("Leukemia with abnormal blood cell growth", "hematological", "tumor"),
    ("Anemia due to iron deficiency", "hematological", "metabolic"),

    # -------- Multisystem --------
    ("Sepsis causing multi organ failure", "multisystemic", "infection"),
    ("Septic shock affecting whole body", "multisystemic", "infection"),

    # -------- Hard Edge Cases --------
    ("Degenerative disc disease of spine", "musculoskeletal", "degenerative"),
    ("Spinal epidural abscess causing paralysis", "neurological", "infection"),
    ("Pulmonary embolism causing breathlessness", "cardiovascular", "vascular"),
]

correct = 0

for text, true_sys, true_type in test_cases:
    pred_sys, pred_type = predict_with_correction(text)

    is_correct = (pred_sys == true_sys) and (pred_type == true_type)

    print("="*70)
    print("TEXT:", text)
    print("PRED:", pred_sys, pred_type)
    print("TRUE:", true_sys, true_type)
    print("CORRECT:", is_correct)

    if is_correct:
        correct += 1

print("\nFINAL ACCURACY:", correct, "/", len(test_cases))
print("ACCURACY %:", (correct/len(test_cases))*100)

TEXT: Myocardial infarction due to blocked coronary artery
PRED: cardiovascular vascular
TRUE: cardiovascular vascular
CORRECT: True
TEXT: Hypertension causing long-term heart damage
PRED: cardiovascular metabolic
TRUE: cardiovascular metabolic
CORRECT: True
TEXT: Atherosclerosis with plaque buildup in arteries
PRED: cardiovascular vascular
TRUE: cardiovascular vascular
CORRECT: True
TEXT: Severe pneumonia with lung infection
PRED: respiratory infection
TRUE: respiratory infection
CORRECT: True
TEXT: Chronic obstructive pulmonary disease (COPD)
PRED: respiratory degenerative
TRUE: respiratory degenerative
CORRECT: True
TEXT: Brain tumor causing seizures
PRED: neurological tumor
TRUE: neurological tumor
CORRECT: True
TEXT: Stroke due to ischemia
PRED: neurological vascular
TRUE: neurological vascular
CORRECT: True
TEXT: Spinal cord injury after trauma
PRED: neurological trauma
TRUE: neurological trauma
CORRECT: True
TEXT: Osteoarthritis causing joint degeneration
PRED: musculoskeletal d

In [ ]:
diverse_tests = [
    # -------- Cardiovascular --------
    ("Patient with chest tightness and elevated troponin levels", "cardiovascular", "vascular"),
    ("Long-standing high blood pressure with organ damage", "cardiovascular", "metabolic"),
    ("Blood clot in pulmonary artery leading to hypoxia", "cardiovascular", "vascular"),

    # -------- Respiratory --------
    ("Chronic smoker with progressive breathing difficulty", "respiratory", "degenerative"),
    ("Viral infection causing inflammation of lungs", "respiratory", "infection"),

    # -------- Neurological --------
    ("Loss of consciousness due to brain hemorrhage", "neurological", "vascular"),
    ("Parkinson disease with tremors and rigidity", "neurological", "degenerative"),
    ("Head trauma leading to brain injury", "neurological", "trauma"),

    # -------- Musculoskeletal --------
    ("Joint pain due to cartilage wear and tear", "musculoskeletal", "degenerative"),
    ("Fracture of arm after fall", "musculoskeletal", "trauma"),
    ("Malignant tumor found in bone tissue", "musculoskeletal", "tumor"),

    # -------- Gastrointestinal --------
    ("Inflammation of appendix causing severe abdominal pain", "gastrointestinal", "infection"),
    ("Cancerous growth in stomach lining", "gastrointestinal", "tumor"),

    # -------- Dermatological --------
    ("Red itchy rash due to allergic reaction", "dermatological", "autoimmune"),
    ("Skin infection with fungal growth", "dermatological", "infection"),

    # -------- Endocrine --------
    ("Elevated blood sugar due to insulin resistance", "endocrine", "metabolic"),
    ("Thyroid hormone overproduction causing weight loss", "endocrine", "metabolic"),

    # -------- Renal --------
    ("Accumulation of waste due to kidney failure", "renal", "metabolic"),
    ("Pain due to crystal formation in urinary tract", "renal", "metabolic"),

    # -------- Immunological --------
    ("Body attacking its own tissues causing inflammation", "immunological", "autoimmune"),
    ("Immune system disorder causing chronic inflammation", "immunological", "autoimmune"),

    # -------- ENT --------
    ("Swelling in jaw muscle treated with botulinum toxin", "ent", "degenerative"),
    ("Sinus inflammation with bacterial infection", "ent", "infection"),

    # -------- Ophthalmic --------
    ("Vision loss due to retinal damage", "ophthalmic", "degenerative"),
    ("Eye redness due to bacterial infection", "ophthalmic", "infection"),

    # -------- Hematological --------
    ("Abnormal proliferation of blood cells", "hematological", "tumor"),
    ("Low hemoglobin due to nutrient deficiency", "hematological", "metabolic"),

    # -------- Multisystem --------
    ("Severe infection spreading through bloodstream affecting organs", "multisystemic", "infection"),
    ("Systemic inflammatory response causing organ failure", "multisystemic", "infection"),

    # -------- HARD EDGE CASES --------
    ("Degeneration of spinal discs causing back pain", "musculoskeletal", "degenerative"),
    ("Infection forming abscess near spinal cord", "neurological", "infection"),
    ("Blockage in lung artery causing breathing issues", "cardiovascular", "vascular"),
    ("Hormonal imbalance affecting metabolism and weight", "endocrine", "metabolic"),
    ("Autoimmune condition affecting multiple joints", "immunological", "autoimmune"),
    ("Severe trauma leading to multiple organ damage", "multisystemic", "trauma"),
    ("Tumor spreading to multiple organs", "multisystemic", "tumor"),
    ("Inflammation of liver due to infection", "hepatobiliary", "infection"),
    ("Gallbladder stones causing abdominal pain", "hepatobiliary", "metabolic"),
    ("Urinary tract infection causing pain", "genitourinary", "infection"),
    ("Prostate enlargement affecting urination", "genitourinary", "degenerative"),
]

correct = 0

for text, true_sys, true_type in diverse_tests:
    pred_sys, pred_type = predict_with_correction(text)

    is_correct = (pred_sys == true_sys) and (pred_type == true_type)

    print("="*70)
    print("TEXT:", text)
    print("PRED:", pred_sys, pred_type)
    print("TRUE:", true_sys, true_type)
    print("CORRECT:", is_correct)

    if is_correct:
        correct += 1

print("\nFINAL ACCURACY:", correct, "/", len(diverse_tests))
print("ACCURACY %:", (correct/len(diverse_tests))*100)

TEXT: Patient with chest tightness and elevated troponin levels
PRED: cardiovascular vascular
TRUE: cardiovascular vascular
CORRECT: True
TEXT: Long-standing high blood pressure with organ damage
PRED: cardiovascular vascular
TRUE: cardiovascular metabolic
CORRECT: False
TEXT: Blood clot in pulmonary artery leading to hypoxia
PRED: cardiovascular vascular
TRUE: cardiovascular vascular
CORRECT: True
TEXT: Chronic smoker with progressive breathing difficulty
PRED: respiratory degenerative
TRUE: respiratory degenerative
CORRECT: True
TEXT: Viral infection causing inflammation of lungs
PRED: respiratory infection
TRUE: respiratory infection
CORRECT: True
TEXT: Loss of consciousness due to brain hemorrhage
PRED: neurological vascular
TRUE: neurological vascular
CORRECT: True
TEXT: Parkinson disease with tremors and rigidity
PRED: neurological degenerative
TRUE: neurological degenerative
CORRECT: True
TEXT: Head trauma leading to brain injury
PRED: neurological trauma
TRUE: neurological trau